In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/__results__.html
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/__huggingface_repos__.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/__notebook__.ipynb
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/__output__.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/custom.css
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/train_results.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/config.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/trainer_state.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/training_args.bin
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/tokenizer.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm/all_results.json
/kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbe

# Init settings

In [2]:
from pathlib import Path
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# PROBE
PROBE_FILE = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "test-on-fabertwikifarsi-making-testset-sec6/"
    "factual_probe/factual_probe_dataset.csv"
)

# ParsBERT
PARSBERT_BASE_MODEL = "HooshvareLab/bert-base-parsbert-uncased"

#  ParsBERT-KG 
PARSBERT_KG_MODEL_PATH = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "8-finetunning-parsbert/parsbert_kg_mlm"
)

OUTPUT_DIR = Path("/kaggle/working/factual_probe_parsbert")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
MAX_LENGTH = 128
TOP_K_TO_SAVE = 10

print("Device:", DEVICE)
print("Probe file exists:", PROBE_FILE.exists())
print("ParsBERT KG model path exists:", PARSBERT_KG_MODEL_PATH.exists())
print("ParsBERT KG model path:", PARSBERT_KG_MODEL_PATH)


Device: cpu
Probe file exists: True
ParsBERT KG model path exists: True
ParsBERT KG model path: /kaggle/input/notebooks/aabdollahii/8-finetunning-parsbert/parsbert_kg_mlm


In [3]:
probe_df = pd.read_csv(
    PROBE_FILE,
    encoding="utf-8-sig",
)

required_columns = {
    "fact_id",
    "subject",
    "predicate",
    "object",
    "prompt",
    "gold_answer",
    "object_token_count",
}

missing_columns = required_columns - set(probe_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

print("Initial shape:", probe_df.shape)
print("Columns:", probe_df.columns.tolist())

display(probe_df.head(10))

evaluation_df = probe_df.copy()

# this code just selects valid Columns
if "valid" in evaluation_df.columns:
    valid_values = (
        evaluation_df["valid"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    )
    evaluation_df = evaluation_df[valid_values].copy()

evaluation_df = evaluation_df[
    evaluation_df["object_token_count"] == 1
].copy()

evaluation_df = evaluation_df.dropna(
    subset=["prompt", "gold_answer"]
).copy()

evaluation_df["prompt"] = (
    evaluation_df["prompt"]
    .astype(str)
    .str.strip()
)

evaluation_df["gold_answer"] = (
    evaluation_df["gold_answer"]
    .astype(str)
    .str.strip()
)

# preprocess 
evaluation_df = evaluation_df.drop_duplicates(
    subset=["fact_id", "prompt", "gold_answer"]
).reset_index(drop=True)

# check only one MASK
mask_counts = evaluation_df["prompt"].str.count(r"\[MASK\]")

invalid_mask_df = evaluation_df[
    mask_counts != 1
].copy()

evaluation_df = evaluation_df[
    mask_counts == 1
].reset_index(drop=True)

print("Final evaluation rows (before ParsBERT re-tokenization):", len(evaluation_df))
print("Rows rejected because mask count was not one:", len(invalid_mask_df))

print("\nPredicate distribution:")
display(
    evaluation_df["predicate"]
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="count")
)


Initial shape: (1206, 15)
Columns: ['fact_id', 'subject', 'predicate', 'object', 'prompt', 'template_id', 'template_type', 'gold_answer', 'gold_token_id', 'object_token_count', 'object_tokens', 'split_type', 'source', 'valid', 'validation_error']


,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid,validation_error
0,fact_000001,محمد پورستار,محل تولد,اردبیل,زادگاه محمد پورستار [MASK] است.,birthplace_03,paraphrased,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
1,fact_000002,بخش ایرندگان,زبان,بلوچی,زبان بخش ایرندگان [MASK] است.,language_01,familiar,بلوچی,31449,1,"[""بلوچی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
2,fact_000003,پرم چوپرا,محل تولد,لاهور,محل تولد پرم چوپرا [MASK] است.,birthplace_01,familiar,لاهور,49461,1,"[""لاهور""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
3,fact_000004,یاروسلاو سایفرت,ملیت,چکی,یاروسلاو سایفرت فردی [MASK] است.,nationality_03,paraphrased,چکی,36644,1,"[""چکی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
4,fact_000005,سه برخوانی,کشور,تهران,کشور محل قرارگیری سه برخوانی، [MASK] است.,country_02,paraphrased,تهران,3148,1,"[""تهران""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
5,fact_000006,پاستورا سولر,ملیت,اسپانیایی,ملیت پاستورا سولر [MASK] است.,nationality_01,familiar,اسپانیایی,17013,1,"[""اسپانیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
6,fact_000007,اسکامپولو ۵۳ (فیلم ۱۹۵۳),زبان,ایتالیایی,زبان اسکامپولو ۵۳ (فیلم ۱۹۵۳) [MASK] است.,language_01,familiar,ایتالیایی,13857,1,"[""ایتالیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
7,fact_000008,قباد آذرآیین,محل تولد,مسجدسلیمان,محل تولد قباد آذرآیین [MASK] است.,birthplace_01,familiar,مسجدسلیمان,32192,1,"[""مسجدسلیمان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
8,fact_000009,ژرژ لامپن,ملیت,فرانسه,ژرژ لامپن فردی [MASK] است.,nationality_03,paraphrased,فرانسه,5916,1,"[""فرانسه""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN
9,fact_000010,کشکش,استان,گیلان,کشکش در استان [MASK] قرار دارد.,province_01,familiar,گیلان,8188,1,"[""گیلان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wiki-data-knowledge-graph-analysis/cleaned_knowledge_graph.csv,True,NaN


Final evaluation rows (before ParsBERT re-tokenization): 1206
Rows rejected because mask count was not one: 0

Predicate distribution:


,predicate,count
0,ملیت,242
1,زبان,240
2,محل تولد,238
3,استان,233
4,کشور,208
5,زبان رسمی,45


- This time we don't rely on the old gold_token_id. Everything aligns from gold_answer and the ParsBERT tokenizer.

In [4]:
parsbert_tokenizer = AutoTokenizer.from_pretrained(
    PARSBERT_BASE_MODEL,
    use_fast=True,
)

print("ParsBERT tokenizer:", parsbert_tokenizer.name_or_path)
print("Vocabulary size:", len(parsbert_tokenizer))
print("Mask token:", parsbert_tokenizer.mask_token)
print("Mask token ID:", parsbert_tokenizer.mask_token_id)

tokenizer = parsbert_tokenizer  # برای استفاده در کل evaluation


config.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

ParsBERT tokenizer: HooshvareLab/bert-base-parsbert-uncased
Vocabulary size: 100000
Mask token: [MASK]
Mask token ID: 3


In [5]:
def tokenize_gold_answer(answer, tokenizer):
    token_ids = tokenizer.encode(
        str(answer),
        add_special_tokens=False,
    )
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    return token_ids, tokens


In [6]:
gold_token_counts = []
gold_token_ids = []
gold_tokens = []

for answer in evaluation_df["gold_answer"]:
    token_ids, tokens = tokenize_gold_answer(
        answer,
        tokenizer,
    )

    gold_token_counts.append(len(token_ids))
    gold_token_ids.append(
        token_ids[0] if len(token_ids) == 1 else None
    )
    gold_tokens.append(tokens)

evaluation_df["parsbert_gold_token_count"] = gold_token_counts
evaluation_df["parsbert_gold_token_id"] = gold_token_ids
evaluation_df["parsbert_gold_tokens"] = [
    json.dumps(tokens, ensure_ascii=False)
    for tokens in gold_tokens
]

new_multi_token_df = evaluation_df[
    evaluation_df["parsbert_gold_token_count"] != 1
].copy()

evaluation_df = evaluation_df[
    evaluation_df["parsbert_gold_token_count"] == 1
].reset_index(drop=True)

evaluation_df["parsbert_gold_token_id"] = (
    evaluation_df["parsbert_gold_token_id"]
    .astype(int)
)

print("Verified single-token rows for ParsBERT:", len(evaluation_df))
print("Rejected after ParsBERT token verification:", len(new_multi_token_df))

if len(new_multi_token_df):
    display(
        new_multi_token_df[
            [
                "gold_answer",
                "parsbert_gold_token_count",
                "parsbert_gold_tokens",
            ]
        ].head(20)
    )


Verified single-token rows for ParsBERT: 1205
Rejected after ParsBERT token verification: 1


,gold_answer,parsbert_gold_token_count,parsbert_gold_tokens
278,تورکی,2,"[""تورک"", ""##ی""]"


In [7]:
evaluation_df["verified_gold_token_id"] = evaluation_df["parsbert_gold_token_id"]
evaluation_df["verified_gold_token_count"] = evaluation_df["parsbert_gold_token_count"]
evaluation_df["verified_gold_tokens"] = evaluation_df["parsbert_gold_tokens"]

print("Final evaluation rows (ParsBERT-compatible):", len(evaluation_df))


Final evaluation rows (ParsBERT-compatible): 1205


# Evaluation fumction 